# Same-checkpoint inference screen

Validation-only T1-T4 and P3 comparison. No competition submission is created.

In [ ]:
import importlib.util
import subprocess
import sys

packages = []
for name, spec in [('ultralytics', 'ultralytics==8.4.147'), ('pycocotools', 'pycocotools')]:
    if importlib.util.find_spec(name) is None:
        packages.append(spec)
if packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *packages], check=True)


In [ ]:
from pathlib import Path
Path('/kaggle/working/hsi_runner.py').write_text('"""Reproducible 16-band hyperspectral detector experiments and submission builder.\n\nSimple explanation:\n\nThis script:\n1. Reads the 16-band hyperspectral images.\n2. Reads the correct object boxes from XML files.\n3. Keeps some training images hidden for validation and holdout testing.\n4. Converts the data into the format YOLO expects.\n5. Trains one YOLO26m or RT-DETR-L object-detection model.\n6. Uses that model on the competition test images.\n7. Creates submission.csv only when holdout mAP50-95 is at least 0.50.\n\nImportant terms:\n- YOLO26m: an object detector. It predicts class, confidence, and box.\n- 16 bands: the 16 spectral channels in each competition image.\n- Validation: hidden training images used to check model progress.\n- Holdout: another hidden local set used as a safer final check.\n- Confidence: how sure the model is about one prediction.\n- Bounding box: rectangle around an object.\n\nThis version can read the competition raw PNG mosaics directly and applies the organizer-provided X2Cube conversion internally. It supports controlled aspect-ratio, cross-validation, spectral-augmentation, and same-checkpoint TTA experiments.\n"""\n\nfrom pathlib import Path\nimport argparse\nimport json\nimport random\nimport hashlib\nimport warnings\nimport importlib.metadata\nimport xml.etree.ElementTree as ET\n\nimport cv2\nfrom PIL import Image\nimport numpy as np\nimport pandas as pd\nimport yaml\nfrom sklearn.model_selection import StratifiedGroupKFold, train_test_split\n\n\n\ndef X2Cube(img, cellSize=4):\n    if img.ndim != 2 or any(n % cellSize for n in img.shape):\n        raise ValueError("Raw image sides must be divisible by 4")\n    B = [cellSize, cellSize]\n    skip = [cellSize, cellSize]\n    M, N = img.shape\n    col_extent = N - B[1] + 1\n    row_extent = M - B[0] + 1\n    start_idx = np.arange(B[0])[:, None] * N + np.arange(B[1])\n    didx = M * N * np.arange(1)\n    start_idx = (didx[:, None] + start_idx.ravel()).reshape((-1, B[0], B[1]))\n    offset_idx = np.arange(row_extent)[:, None] * N + np.arange(col_extent)\n    out = np.take(\n        img,\n        start_idx.ravel()[:, None] + offset_idx[::skip[0], ::skip[1]].ravel(),\n    )\n    out = np.transpose(out)\n    return out.reshape(M // cellSize, N // cellSize, cellSize * cellSize)\n\n\ndef read_classes(path):\n    names = [x.strip() for x in Path(path).read_text().splitlines() if x.strip()]\n    if len(names) != 18:\n        raise ValueError(f"Expected 18 classes, found {len(names)}")\n    return names\n\n\ndef read_cube(path):\n    path = Path(path)\n    if path.suffix.lower() == ".png":\n        raw = np.array(Image.open(path))\n        if raw.ndim != 2:\n            raise ValueError(f"{path} must be a 2D raw mosaic PNG, got {raw.shape}")\n        x = X2Cube(raw)\n    elif path.suffix.lower() == ".npy":\n        x = np.load(path)\n    elif path.suffix.lower() == ".npz":\n        z = np.load(path)\n        x = z[z.files[0]]\n    elif path.suffix.lower() in {".tif", ".tiff"}:\n        ok, frames = cv2.imreadmulti(str(path), flags=cv2.IMREAD_UNCHANGED)\n        if not ok:\n            raise ValueError(f"Cannot read {path}")\n        x = np.stack(frames, axis=-1)\n    else:\n        raise ValueError(f"Unsupported cube format: {path.suffix}")\n    x = np.asarray(x)\n    if not np.isfinite(x).all():\n        raise ValueError(f"Non-finite image values in {path}")\n    if x.ndim != 3:\n        raise ValueError(f"{path} must be 3D, got {x.shape}")\n    if x.shape[-1] == 16:\n        return x.astype(np.float32)\n    if x.shape[0] == 16:\n        return np.moveaxis(x, 0, -1).astype(np.float32)\n    raise ValueError(f"{path} does not contain 16 bands: {x.shape}")\n\n\ndef collect_cubes(root):\n    root = Path(root)\n    files = []\n    for ext in ("*.png", "*.npy", "*.npz", "*.tif", "*.tiff"):\n        files.extend(root.rglob(ext))\n    files = sorted({p.resolve() for p in files})\n    if not files:\n        raise FileNotFoundError(\n            "No 16-band cubes found. Run the organizer-compatible X2Cube conversion first."\n        )\n    return files\n\n\ndef band_limits(files, samples_per_image=256, seed=42):\n    rng = np.random.default_rng(seed)\n    chunks = []\n    for p in files:\n        x = read_cube(p).reshape(-1, 16)\n        n = min(samples_per_image, len(x))\n        idx = rng.choice(len(x), n, replace=False)\n        chunks.append(x[idx])\n    sample = np.concatenate(chunks, axis=0)\n    lo = np.percentile(sample, 0.5, axis=0)\n    hi = np.percentile(sample, 99.5, axis=0)\n    hi = np.maximum(hi, lo + 1e-6)\n    return lo, hi\n\n\ndef model_geometry(height, width, input_policy):\n    if input_policy == "native":\n        return {\n            "original_height": height,\n            "original_width": width,\n            "model_height": height,\n            "model_width": width,\n            "pad_top": 0,\n            "pad_left": 0,\n        }\n    if input_policy != "square_pad":\n        raise ValueError(f"Unknown input policy: {input_policy}")\n    side = max(height, width)\n    return {\n        "original_height": height,\n        "original_width": width,\n        "model_height": side,\n        "model_width": side,\n        "pad_top": (side - height) // 2,\n        "pad_left": (side - width) // 2,\n    }\n\n\ndef resolve_input_policy(family, input_policy):\n    if input_policy != "auto":\n        return input_policy\n    return "square_pad" if family == "rtdetr" else "native"\n\n\ndef prepare_cube(cube, lo, hi, input_policy, pipeline_state=None):\n    if pipeline_state is None:\n        x = np.clip((cube - lo) / (hi - lo), 0, 1)\n        x = np.rint(x * 255).astype(np.uint8)\n    else:\n        try:\n            from Experiments.preprocessing.pipeline import apply_pipeline\n        except ModuleNotFoundError:\n            from pipeline import apply_pipeline\n        x = apply_pipeline(cube, pipeline_state, output="uint8")\n    height, width = x.shape[:2]\n    geometry = model_geometry(height, width, input_policy)\n    if input_policy == "square_pad":\n        padded = np.zeros(\n            (geometry["model_height"], geometry["model_width"], 16),\n            dtype=np.uint8,\n        )\n        top = geometry["pad_top"]\n        left = geometry["pad_left"]\n        padded[top : top + height, left : left + width] = x\n        x = padded\n    return x, geometry\n\n\ndef write_tiff(cube, path, lo, hi, input_policy, pipeline_state=None):\n    x, geometry = prepare_cube(cube, lo, hi, input_policy, pipeline_state)\n    chw = np.moveaxis(x, -1, 0)\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    ok = cv2.imwritemulti(str(path), chw)\n    if not ok:\n        raise IOError(f"Failed to write {path}")\n    return geometry\n\n\ndef transform_rows(rows, geometry):\n    source_width = geometry["original_width"]\n    source_height = geometry["original_height"]\n    target_width = geometry["model_width"]\n    target_height = geometry["model_height"]\n    pad_left = geometry["pad_left"]\n    pad_top = geometry["pad_top"]\n    transformed = []\n    for cls, cx, cy, bw, bh in rows:\n        transformed.append(\n            (\n                cls,\n                (cx * source_width + pad_left) / target_width,\n                (cy * source_height + pad_top) / target_height,\n                bw * source_width / target_width,\n                bh * source_height / target_height,\n            )\n        )\n    return transformed\n\n\ndef restore_boxes(boxes, geometry):\n    boxes = np.asarray(boxes, dtype=np.float32).copy()\n    boxes[:, [0, 2]] -= geometry["pad_left"]\n    boxes[:, [1, 3]] -= geometry["pad_top"]\n    boxes[:, [0, 2]] = np.clip(boxes[:, [0, 2]], 0, geometry["original_width"])\n    boxes[:, [1, 3]] = np.clip(boxes[:, [1, 3]], 0, geometry["original_height"])\n    return boxes\n\n\ndef spectral_augment(cube, rng, gain_limit, tilt_limit, noise_std):\n    cube = cube.astype(np.float32, copy=True)\n    gain = rng.uniform(1.0 - gain_limit, 1.0 + gain_limit)\n    tilt = rng.uniform(-tilt_limit, tilt_limit)\n    wavelength_axis = np.linspace(-1.0, 1.0, 16, dtype=np.float32)\n    cube *= gain * (1.0 + tilt * wavelength_axis[None, None, :])\n    if noise_std > 0:\n        scale = np.maximum(cube.std(axis=(0, 1)), 1e-6)\n        noise = rng.normal(0.0, noise_std, size=cube.shape).astype(np.float32)\n        cube += noise * scale[None, None, :]\n    return cube\n\n\ndef stable_rng(seed, stem, copy_index):\n    digest = hashlib.sha256(f"{seed}:{stem}:{copy_index}".encode()).digest()\n    return np.random.default_rng(int.from_bytes(digest[:8], "little"))\n\n\ndef expand_rgb_stem(target_model, source_model):\n    import torch\n\n    target_layers = [layer for layer in target_model.modules() if isinstance(layer, torch.nn.Conv2d)]\n    source_layers = [layer for layer in source_model.modules() if isinstance(layer, torch.nn.Conv2d)]\n    target = next((layer for layer in target_layers if layer.in_channels == 16), None)\n    source = next(\n        (\n            layer\n            for layer in source_layers\n            if layer.in_channels == 3\n            and target is not None\n            and layer.out_channels == target.out_channels\n            and layer.kernel_size == target.kernel_size\n        ),\n        None,\n    )\n    if target is None or source is None:\n        raise RuntimeError("Could not identify compatible 3-channel and 16-channel input stems")\n    expanded = source.weight.detach().mean(dim=1, keepdim=True).repeat(1, 16, 1, 1)\n    expanded *= 3.0 / 16.0\n    target.weight.data.copy_(expanded.to(device=target.weight.device, dtype=target.weight.dtype))\n    if target.bias is not None and source.bias is not None:\n        target.bias.data.copy_(source.bias.detach().to(device=target.bias.device, dtype=target.bias.dtype))\n    print("Initialized the 16-channel input stem from RGB weights with activation-scale correction")\n\n\ndef trainer_class(args):\n    if args.stem_init == "random":\n        return None\n    if args.family == "rtdetr":\n        from ultralytics.models.rtdetr.train import RTDETRTrainer\n\n        class HSITrainer(RTDETRTrainer):\n            def get_model(self, cfg=None, weights=None, verbose=True):\n                model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)\n                if weights is None:\n                    raise RuntimeError("rgb_mean stem initialization requires pretrained weights")\n                expand_rgb_stem(model, weights)\n                return model\n\n        return HSITrainer\n    from ultralytics.models.yolo.detect import DetectionTrainer\n\n    class HSITrainer(DetectionTrainer):\n        def get_model(self, cfg=None, weights=None, verbose=True):\n            model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)\n            if weights is None:\n                raise RuntimeError("rgb_mean stem initialization requires pretrained weights")\n            expand_rgb_stem(model, weights)\n            return model\n\n    return HSITrainer\n\n\ndef parse_voc(xml_path, class_to_id):\n    root = ET.parse(xml_path).getroot()\n    size = root.find("size")\n    width = int(float(size.findtext("width")))\n    height = int(float(size.findtext("height")))\n    rows = []\n    for obj in root.findall("object"):\n        name = obj.findtext("name").strip()\n        if name not in class_to_id:\n            raise ValueError(f"Unknown class {name} in {xml_path}")\n        b = obj.find("bndbox")\n        x1 = float(b.findtext("xmin"))\n        y1 = float(b.findtext("ymin"))\n        x2 = float(b.findtext("xmax"))\n        y2 = float(b.findtext("ymax"))\n        x1 = min(max(x1, 0.0), width)\n        x2 = min(max(x2, 0.0), width)\n        y1 = min(max(y1, 0.0), height)\n        y2 = min(max(y2, 0.0), height)\n        if x2 <= x1 or y2 <= y1:\n            continue\n        cx = ((x1 + x2) / 2) / width\n        cy = ((y1 + y2) / 2) / height\n        bw = (x2 - x1) / width\n        bh = (y2 - y1) / height\n        rows.append((class_to_id[name], cx, cy, bw, bh))\n    return width, height, rows\n\n\ndef annotation_map(xml_root):\n    xmls = list(Path(xml_root).rglob("*.xml"))\n    if not xmls:\n        raise FileNotFoundError(f"No VOC XML files found under {xml_root}")\n    return unique_stems(xmls)\n\n\ndef rarest_image_label(stems, boxes_by_stem):\n    counts = {}\n    for stem in stems:\n        for row in boxes_by_stem[stem]:\n            cls = int(row[0])\n            counts[cls] = counts.get(cls, 0) + 1\n    labels = []\n    for stem in stems:\n        classes = {int(row[0]) for row in boxes_by_stem[stem]}\n        if not classes:\n            labels.append(-1)\n        else:\n            labels.append(min(classes, key=lambda c: counts[c]))\n    return labels\n\n\ndef safe_split(stems, boxes_by_stem, seed):\n    labels = rarest_image_label(stems, boxes_by_stem)\n    try:\n        train, temp = train_test_split(\n            stems,\n            test_size=0.20,\n            random_state=seed,\n            stratify=labels,\n        )\n        temp_labels = rarest_image_label(temp, boxes_by_stem)\n        val, holdout = train_test_split(\n            temp,\n            test_size=0.50,\n            random_state=seed,\n            stratify=temp_labels,\n        )\n    except ValueError as exc:\n        warnings.warn(f"Class-balanced split failed; using random split: {exc}")\n        train, temp = train_test_split(\n            stems,\n            test_size=0.20,\n            random_state=seed,\n            shuffle=True,\n        )\n        val, holdout = train_test_split(\n            temp,\n            test_size=0.50,\n            random_state=seed,\n            shuffle=True,\n        )\n    return sorted(train), sorted(val), sorted(holdout)\n\n\ndef read_groups(path, stems):\n    if not path:\n        return {stem: stem for stem in stems}, "image_id_only"\n    frame = pd.read_csv(path, dtype=str)\n    expected = {"image_id", "group_id"}\n    if set(frame.columns) != expected:\n        raise ValueError(f"Groups file columns must be {sorted(expected)}")\n    if frame["image_id"].duplicated().any():\n        raise ValueError("Groups file contains duplicate image_id values")\n    groups = dict(zip(frame["image_id"], frame["group_id"]))\n    missing = sorted(set(stems) - set(groups))\n    if missing:\n        raise ValueError(f"Groups file is missing {len(missing)} training IDs")\n    return {stem: groups[stem] for stem in stems}, str(Path(path))\n\n\ndef cv_split(stems, boxes_by_stem, groups_by_stem, seed, folds, fold_index):\n    if folds < 2:\n        raise ValueError("cv_folds must be at least 2")\n    if not 0 <= fold_index < folds:\n        raise ValueError(f"cv_fold_index must be in 0..{folds - 1}")\n    labels = np.asarray(rarest_image_label(stems, boxes_by_stem))\n    groups = np.asarray([groups_by_stem[stem] for stem in stems])\n    holdout_splitter = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=seed)\n    design_idx, holdout_idx = next(holdout_splitter.split(stems, labels, groups))\n    design_stems = np.asarray(stems)[design_idx]\n    design_labels = labels[design_idx]\n    design_groups = groups[design_idx]\n    fold_splitter = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=seed + 1)\n    selected = list(fold_splitter.split(design_stems, design_labels, design_groups))[fold_index]\n    train_idx, val_idx = selected\n    return (\n        sorted(design_stems[train_idx].tolist()),\n        sorted(design_stems[val_idx].tolist()),\n        sorted(np.asarray(stems)[holdout_idx].tolist()),\n    )\n\n\ndef load_split_manifest(path, stems):\n    split_map = json.loads(Path(path).read_text())\n    if set(split_map) != {"train", "val", "test"}:\n        raise ValueError("Split manifest must contain train, val, and test")\n    flattened = [str(stem) for split in split_map.values() for stem in split]\n    if len(flattened) != len(set(flattened)):\n        raise ValueError("Split manifest contains duplicate IDs across splits")\n    if set(flattened) != set(stems):\n        raise ValueError("Split manifest IDs do not exactly match usable training IDs")\n    return {key: sorted(map(str, value)) for key, value in split_map.items()}\n\n\ndef assert_group_isolation(split_map, groups_by_stem):\n    group_splits = {}\n    for split, stems in split_map.items():\n        for stem in stems:\n            group = groups_by_stem[stem]\n            if group in group_splits and group_splits[group] != split:\n                raise ValueError(f"Group {group} leaks across {group_splits[group]} and {split}")\n            group_splits[group] = split\n\n\ndef write_label(path, rows):\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    text = "\\n".join(\n        f"{int(c)} {cx:.8f} {cy:.8f} {w:.8f} {h:.8f}"\n        for c, cx, cy, w, h in rows\n    )\n    Path(path).write_text(text + ("\\n" if text else ""))\n\n\ndef image_id_value(stem):\n    return int(stem) if stem.isdigit() else stem\n\n\ndef parse_class_ids(value):\n    if not value:\n        return []\n    ids = sorted({int(item) for item in value.split(",")})\n    if any(class_id < 0 or class_id > 17 for class_id in ids):\n        raise argparse.ArgumentTypeError("Class IDs must be between 0 and 17")\n    return ids\n\n\ndef build_dataset(args):\n    work = Path(args.workdir)\n    cache_root = Path(args.cache_dir) if args.cache_dir else work\n    prepared = cache_root / "dataset"\n    if prepared.exists():\n        raise FileExistsError(f"Use a fresh workdir; keeping existing data: {prepared}")\n    names = read_classes(args.class_file)\n    class_to_id = {name: i for i, name in enumerate(names)}\n    train_cubes = collect_cubes(args.train_cubes)\n    test_cubes = collect_cubes(args.test_cubes)\n    ranking_cubes = collect_cubes(args.ranking_cubes) if args.ranking_cubes else []\n    xmls = annotation_map(args.train_xml)\n    cube_by_stem = unique_stems(train_cubes)\n    inference_by_stem = unique_stems(test_cubes + ranking_cubes)\n    if set(cube_by_stem) != set(xmls):\n        raise ValueError("Training image and XML IDs do not match")\n    stems = sorted(set(cube_by_stem) & set(xmls))\n    bad_stems = {"1227", "1836", "1855"}\n    stems = [s for s in stems if s not in bad_stems]\n    print(f"Skipping mismatched images: {sorted(bad_stems)}")\n\n    if not stems:\n        raise RuntimeError("No matching training cube/XML stems found")\n\n    boxes_by_stem = {}\n    dims_by_stem = {}\n    for stem in stems:\n        w, h, rows = parse_voc(xmls[stem], class_to_id)\n        boxes_by_stem[stem] = rows\n        dims_by_stem[stem] = (w, h)\n\n    groups_by_stem, groups_source = read_groups(args.groups_file, stems)\n    if args.split_manifest:\n        split_map = load_split_manifest(args.split_manifest, stems)\n    elif args.cv_fold_index is not None:\n        train_stems, val_stems, holdout_stems = cv_split(\n            stems,\n            boxes_by_stem,\n            groups_by_stem,\n            args.seed,\n            args.cv_folds,\n            args.cv_fold_index,\n        )\n        split_map = {"train": train_stems, "val": val_stems, "test": holdout_stems}\n    else:\n        train_stems, val_stems, holdout_stems = safe_split(stems, boxes_by_stem, args.seed)\n        split_map = {"train": train_stems, "val": val_stems, "test": holdout_stems}\n    assert_group_isolation(split_map, groups_by_stem)\n    train_stems = split_map["train"]\n    val_stems = split_map["val"]\n    holdout_stems = split_map["test"]\n    normalization = getattr(args, "normalization", "N0")\n    spectral_preprocess = getattr(args, "spectral_preprocess", "SP0")\n    pipeline_state = None\n    if normalization == "N0" and spectral_preprocess == "SP0":\n        lo, hi = band_limits([cube_by_stem[s] for s in train_stems], seed=args.seed)\n    else:\n        try:\n            from Experiments.preprocessing.pipeline import fit_pipeline, save_pipeline_state\n        except ModuleNotFoundError:\n            from pipeline import fit_pipeline, save_pipeline_state\n        pipeline_state = fit_pipeline(\n            (read_cube(cube_by_stem[s]) for s in train_stems),\n            normalization,\n            spectral_method=spectral_preprocess,\n            seed=args.seed,\n            samples_per_cube=getattr(args, "normalization_samples_per_image", 256),\n            mnf_components=getattr(args, "mnf_components", None),\n            savgol_window=getattr(args, "savgol_window", 5),\n            savgol_order=getattr(args, "savgol_order", 2),\n        )\n        lo = hi = None\n        save_pipeline_state(pipeline_state, work / "spectral_pipeline_state.json")\n    input_policy = resolve_input_policy(args.family, args.input_policy)\n    work.mkdir(parents=True, exist_ok=True)\n    (work / "split_manifest.json").write_text(json.dumps(split_map, indent=2))\n    train_images_written = 0\n    for split, split_stems in split_map.items():\n        for stem in split_stems:\n            cube = read_cube(cube_by_stem[stem])\n            h, w = cube.shape[:2]\n            ann_w, ann_h = dims_by_stem[stem]\n            if (w, h) != (ann_w, ann_h):\n                raise ValueError(\n                    f"Spatial mismatch for {stem}: cube {(w, h)} vs annotation {(ann_w, ann_h)}"\n                )\n            geometry = write_tiff(\n                cube,\n                prepared / "images" / split / f"{stem}.tiff",\n                lo,\n                hi,\n                input_policy,\n                pipeline_state,\n            )\n            rows = transform_rows(boxes_by_stem[stem], geometry)\n            write_label(prepared / "labels" / split / f"{stem}.txt", rows)\n            if split != "train":\n                continue\n            train_images_written += 1\n            classes = {int(row[0]) for row in boxes_by_stem[stem]}\n            selected = not args.spectral_aug_classes or classes.intersection(args.spectral_aug_classes)\n            if not selected:\n                continue\n            for copy_index in range(args.spectral_aug_copies):\n                suffix = f"{stem}_hsaug{copy_index + 1}"\n                augmented = spectral_augment(\n                    cube,\n                    stable_rng(args.seed, stem, copy_index),\n                    args.spectral_gain,\n                    args.spectral_tilt,\n                    args.spectral_noise,\n                )\n                write_tiff(\n                    augmented,\n                    prepared / "images" / split / f"{suffix}.tiff",\n                    lo,\n                    hi,\n                    input_policy,\n                    pipeline_state,\n                )\n                write_label(prepared / "labels" / split / f"{suffix}.txt", rows)\n                train_images_written += 1\n\n    test_out = cache_root / "test_tiff"\n    if test_out.exists():\n        raise FileExistsError(f"Use a fresh workdir; keeping existing data: {test_out}")\n    test_out.mkdir(parents=True, exist_ok=True)\n    inference_geometry = {}\n    for stem, path in inference_by_stem.items():\n        inference_geometry[stem] = write_tiff(\n            read_cube(path),\n            test_out / f"{stem}.tiff",\n            lo,\n            hi,\n            input_policy,\n            pipeline_state,\n        )\n    (work / "inference_geometry.json").write_text(json.dumps(inference_geometry, indent=2))\n\n    data = {\n        "path": str(prepared),\n        "train": "images/train",\n        "val": "images/val",\n        "test": "images/test",\n        "channels": 16,\n        "names": {i: n for i, n in enumerate(names)},\n    }\n    yaml_path = work / "dataset.yaml"\n    yaml_path.write_text(yaml.safe_dump(data, sort_keys=False))\n\n    report = {\n        "train_images_total": len(stems),\n        "train_split": len(train_stems),\n        "val_split": len(val_stems),\n        "holdout_split": len(holdout_stems),\n        "prepared_train_images": train_images_written,\n        "competition_test_images": len(test_cubes),\n        "competition_ranking_images": len(ranking_cubes),\n        "class_names": names,\n        "excluded_ids": sorted(bad_stems),\n        "split_ids": split_map,\n        "split_mode": "manifest" if args.split_manifest else ("cv" if args.cv_fold_index is not None else "legacy"),\n        "cv_folds": args.cv_folds if args.cv_fold_index is not None else None,\n        "cv_fold_index": args.cv_fold_index,\n        "groups_source": groups_source,\n        "group_count": len(set(groups_by_stem.values())),\n        "input_policy": input_policy,\n        "spectral_augmentation": {\n            "copies": args.spectral_aug_copies,\n            "classes": args.spectral_aug_classes,\n            "gain_limit": args.spectral_gain,\n            "tilt_limit": args.spectral_tilt,\n            "noise_std": args.spectral_noise,\n        },\n        "class_counts": {split: np.bincount([int(row[0]) for stem in ids for row in boxes_by_stem[stem]], minlength=18).tolist() for split, ids in split_map.items()},\n        "band_low_0_5pct": lo.tolist() if lo is not None else None,\n        "band_high_99_5pct": hi.tolist() if hi is not None else None,\n        "normalization": normalization,\n        "spectral_preprocess": spectral_preprocess,\n        "spectral_pipeline_state": pipeline_state.to_dict() if pipeline_state is not None else None,\n    }\n    (work / "data_audit.json").write_text(json.dumps(report, indent=2))\n    return yaml_path, test_out, inference_geometry\n\n\ndef train(args, yaml_path):\n    model = model_class(args)(args.model)\n    result = model.train(\n        trainer=trainer_class(args),\n        data=str(yaml_path),\n        epochs=args.epochs,\n        imgsz=args.imgsz,\n        batch=args.batch,\n        device=args.device,\n        workers=args.workers,\n        project=str(Path(args.workdir) / "runs"),\n        name=args.run_name,\n        seed=args.seed,\n        deterministic=args.family == "yolo",\n        amp=args.family == "yolo",\n        rect=args.family == "yolo",\n        multi_scale=args.multi_scale,\n        patience=25,\n        close_mosaic=15,\n        hsv_h=0.0,\n        hsv_s=0.0,\n        hsv_v=0.0,\n        degrees=5.0,\n        translate=0.08,\n        scale=0.35,\n        fliplr=0.5,\n        flipud=0.0,\n        mosaic=0.5,\n        mixup=0.0,\n        copy_paste=0.0,\n        augmentations=[],\n        plots=False,\n        save=True,\n        verbose=True,\n        pretrained=True,\n    )\n    best = Path(result.save_dir) / "weights" / "best.pt"\n    if not best.exists():\n        raise FileNotFoundError(best)\n    return best\n\n\ndef localization_finetune(args, yaml_path, checkpoint):\n    if args.localization_epochs <= 0:\n        return None\n    model = model_class(args)(str(checkpoint))\n    result = model.train(\n        data=str(yaml_path),\n        epochs=args.localization_epochs,\n        imgsz=args.imgsz,\n        batch=args.batch,\n        device=args.device,\n        workers=args.workers,\n        project=str(Path(args.workdir) / "runs"),\n        name=f"{args.run_name}_localize",\n        seed=args.seed,\n        deterministic=args.family == "yolo",\n        amp=args.family == "yolo",\n        rect=args.family == "yolo",\n        multi_scale=0.0,\n        patience=args.localization_epochs,\n        lr0=args.localization_lr0,\n        lrf=0.1,\n        cos_lr=True,\n        warmup_epochs=0.5,\n        close_mosaic=0,\n        hsv_h=0.0,\n        hsv_s=0.0,\n        hsv_v=0.0,\n        degrees=0.0,\n        translate=0.02,\n        scale=0.10,\n        fliplr=0.5,\n        flipud=0.0,\n        mosaic=0.0,\n        mixup=0.0,\n        copy_paste=0.0,\n        augmentations=[],\n        plots=False,\n        save=True,\n        verbose=True,\n        pretrained=True,\n    )\n    best = Path(result.save_dir) / "weights" / "best.pt"\n    if not best.exists():\n        raise FileNotFoundError(best)\n    return best\n\n\ndef predict(args, best, test_out, inference_geometry, tta):\n    model = model_class(args)(str(best))\n    results = model.predict(\n        source=str(test_out),\n        imgsz=args.imgsz,\n        conf=args.predict_conf,\n        iou=args.iou,\n        max_det=args.predict_max_det,\n        device=args.device,\n        augment=tta,\n        stream=True,\n        verbose=False,\n    )\n    rows = []\n    seen = set()\n    empty = []\n    for r in results:\n        seen.add(Path(r.path).stem)\n        if r.boxes is None or len(r.boxes) == 0:\n            empty.append(Path(r.path).stem)\n        stem = Path(r.path).stem\n        if r.boxes is None:\n            continue\n        xyxy = restore_boxes(\n            r.boxes.xyxy.detach().cpu().numpy(),\n            inference_geometry[stem],\n        )\n        confs = r.boxes.conf.detach().cpu().numpy()\n        classes = r.boxes.cls.detach().cpu().numpy().astype(int)\n        for box, score, cls in zip(xyxy, confs, classes):\n            x1, y1, x2, y2 = map(float, box)\n            if x2 <= x1 or y2 <= y1:\n                continue\n            rows.append(\n                {\n                    "image_id": image_id_value(stem),\n                    "class_id": int(cls),\n                    "confidence": float(score),\n                    "x1": x1,\n                    "y1": y1,\n                    "x2": x2,\n                    "y2": y2,\n                }\n            )\n\n    expected_ids = set(inference_geometry)\n    if seen != expected_ids:\n        raise RuntimeError("Some test images were not processed")\n    counts = pd.Series([row["image_id"] for row in rows]).value_counts()\n    (Path(args.workdir) / "prediction_audit.json").write_text(json.dumps({\n        "processed_images": len(seen),\n        "no_detection_ids": empty,\n        "prediction_confidence": args.predict_conf,\n        "prediction_max_det": args.predict_max_det,\n        "tta": tta,\n        "predictions_total": len(rows),\n        "predictions_per_image_min": int(counts.min()) if len(counts) else 0,\n        "predictions_per_image_median": float(counts.median()) if len(counts) else 0.0,\n        "predictions_per_image_max": int(counts.max()) if len(counts) else 0,\n    }, indent=2))\n    sub = pd.DataFrame(rows)\n    if sub.empty:\n        raise RuntimeError("No detections produced")\n    sub.insert(0, "id", np.arange(len(sub), dtype=int))\n    expected = ["id", "image_id", "class_id", "confidence", "x1", "y1", "x2", "y2"]\n    sub = sub[expected]\n    if args.sample_submission and pd.read_csv(args.sample_submission, nrows=0).columns.tolist() != expected:\n        raise ValueError("Sample submission columns do not match")\n    if not np.isfinite(sub[["confidence", "x1", "y1", "x2", "y2"]].to_numpy()).all():\n        raise ValueError("Non-finite submission values")\n    if not sub["id"].is_unique:\n        raise RuntimeError("Submission id is not unique")\n    if not sub["class_id"].between(0, 17).all():\n        raise RuntimeError("class_id outside 0..17")\n    if not sub["confidence"].between(0, 1).all():\n        raise RuntimeError("confidence outside 0..1")\n    if not ((sub["x2"] > sub["x1"]) & (sub["y2"] > sub["y1"])).all():\n        raise RuntimeError("Invalid boxes")\n    out_path = Path(args.workdir) / "submission.csv"\n    sub.to_csv(out_path, index=False)\n    return out_path\n\n\ndef unique_stems(files):\n    result = {}\n    for path in files:\n        if path.stem in result:\n            raise ValueError(f"Duplicate image or XML ID: {path.stem}")\n        result[path.stem] = path\n    return result\n\n\ndef model_class(args):\n    from ultralytics import RTDETR, YOLO\n\n    return RTDETR if args.family == "rtdetr" else YOLO\n\n\ndef metric_summary(metrics):\n    class_ids = [int(c) for c in metrics.box.ap_class_index]\n    per_class = {}\n    for position, class_id in enumerate(class_ids):\n        per_class[str(class_id)] = {\n            "ap50_95": float(metrics.box.ap[position]),\n            "ap50": float(metrics.box.ap50[position]),\n            "ap75": float(metrics.box.all_ap[position, 5]),\n            "precision": float(metrics.box.p[position]),\n            "recall": float(metrics.box.r[position]),\n        }\n    summary = {\n        "map50_95": float(metrics.box.map),\n        "map50": float(metrics.box.map50),\n        "map75": float(metrics.box.map75),\n        "localization_gap_map50_minus_map75": float(metrics.box.map50 - metrics.box.map75),\n        "per_class": per_class,\n        "worst_present_class_ap": min((row["ap50_95"] for row in per_class.values()), default=None),\n        "absent_class_ids": sorted(set(range(18)) - set(class_ids)),\n    }\n    matrix = np.asarray(metrics.confusion_matrix.matrix)\n    if matrix.shape == (19, 19):\n        true_positive = np.diag(matrix)[:18]\n        summary["false_positives_per_class"] = (\n            matrix[:18, :].sum(axis=1) - true_positive\n        ).astype(int).tolist()\n        summary["false_negatives_per_class"] = (\n            matrix[:, :18].sum(axis=0) - true_positive\n        ).astype(int).tolist()\n        summary["confusion_matrix_predicted_rows_true_columns"] = matrix.astype(int).tolist()\n    values = [summary["map50_95"], summary["map50"], summary["map75"]]\n    if not np.isfinite(values).all():\n        raise RuntimeError("Non-finite evaluation metrics")\n    return summary\n\n\ndef evaluate(args, best, yaml_path, splits, output_name, tta=False):\n    model = model_class(args)(str(best))\n    report = {\n        "checkpoint": str(best),\n        "sha256": hashlib.sha256(Path(best).read_bytes()).hexdigest(),\n        "tta": tta,\n    }\n    for split in splits:\n        metrics = model.val(data=str(yaml_path), split=split, imgsz=args.imgsz,\n                            batch=args.batch if args.batch > 0 else 4,\n                            device=args.device, workers=args.workers, conf=args.conf,\n                            max_det=args.max_det, iou=args.iou, plots=False,\n                            augment=tta,\n                            project=str(Path(args.workdir) / "evaluation"),\n                            name=f"{Path(output_name).stem}_{split}")\n        report[split] = metric_summary(metrics)\n        (Path(args.workdir) / output_name).write_text(json.dumps(report, indent=2))\n    return report\n\n\ndef promote_candidate(base_report, candidate_report, min_gain):\n    base = base_report["val"]\n    candidate = candidate_report["val"]\n    gain = candidate["map50_95"] - base["map50_95"]\n    map75_gain = candidate["map75"] - base["map75"]\n    worst_regression = candidate["worst_present_class_ap"] - base["worst_present_class_ap"]\n    passed = (gain >= min_gain or (gain >= 0 and map75_gain >= min_gain)) and worst_regression >= -0.03\n    return {\n        "passed": passed,\n        "map50_95_gain": gain,\n        "map75_gain": map75_gain,\n        "worst_present_class_ap_change": worst_regression,\n        "minimum_gain": min_gain,\n    }\n\n\ndef check_submission_gate(args, report):\n    holdout_map = report["test"]["map50_95"]\n    gate = {\n        "metric": "holdout_mAP50_95",\n        "value": holdout_map,\n        "minimum": args.min_holdout_map,\n        "passed": holdout_map >= args.min_holdout_map,\n    }\n    (Path(args.workdir) / "submission_gate.json").write_text(json.dumps(gate, indent=2))\n    return gate\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--train-cubes", required=True)\n    p.add_argument("--test-cubes", required=True)\n    p.add_argument("--ranking-cubes")\n    p.add_argument("--train-xml", required=True)\n    p.add_argument("--class-file", required=True)\n    p.add_argument("--workdir", default="/kaggle/working/hsi_submission01")\n    p.add_argument("--cache-dir")\n    p.add_argument("--groups-file")\n    p.add_argument("--split-manifest")\n    p.add_argument("--cv-folds", type=int, default=3)\n    p.add_argument("--cv-fold-index", type=int)\n    p.add_argument("--model", default="yolo26m.pt")\n    p.add_argument("--checkpoint")\n    p.add_argument("--epochs", type=int, default=120)\n    p.add_argument("--imgsz", type=int, default=1024)\n    p.add_argument("--workers", type=int, default=2)\n    p.add_argument("--device", default="0")\n    p.add_argument("--seed", type=int, default=42)\n    p.add_argument("--input-policy", choices=["auto", "native", "square_pad"], default="auto")\n    p.add_argument("--stem-init", choices=["random", "rgb_mean"], default="rgb_mean")\n    p.add_argument("--normalization", choices=["N0", "N1", "N2", "N3", "N4"], default="N0")\n    p.add_argument("--spectral-preprocess", choices=["SP0", "SP1", "SP2", "SP3"], default="SP0")\n    p.add_argument("--normalization-samples-per-image", type=int, default=256)\n    p.add_argument("--mnf-components", type=int)\n    p.add_argument("--savgol-window", type=int, default=5)\n    p.add_argument("--savgol-order", type=int, default=2)\n    p.add_argument("--multi-scale", type=float, default=0.0)\n    p.add_argument("--spectral-aug-copies", type=int, default=0)\n    p.add_argument("--spectral-aug-classes", type=parse_class_ids, default=[])\n    p.add_argument("--spectral-gain", type=float, default=0.05)\n    p.add_argument("--spectral-tilt", type=float, default=0.03)\n    p.add_argument("--spectral-noise", type=float, default=0.01)\n    p.add_argument("--localization-epochs", type=int, default=0)\n    p.add_argument("--localization-lr0", type=float, default=0.0001)\n    p.add_argument("--promotion-min-gain", type=float, default=0.005)\n    p.add_argument("--tta", action="store_true")\n    p.add_argument("--tta-min-gain", type=float, default=0.002)\n    p.add_argument("--design-only", action="store_true")\n    p.add_argument("--conf", type=float, default=0.001)\n    p.add_argument("--predict-conf", type=float, default=0.001)\n    p.add_argument("--iou", type=float, default=0.65)\n    p.add_argument("--max-det", type=int, default=300)\n    p.add_argument("--predict-max-det", type=int, default=300)\n    p.add_argument("--min-holdout-map", type=float, default=0.50)\n    p.add_argument("--family", choices=["yolo", "rtdetr"], default="yolo")\n    p.add_argument("--batch", type=int, default=-1)\n    p.add_argument("--run-name", default="submission01")\n    p.add_argument("--sample-submission")\n    p.add_argument("--evaluate-holdout", action="store_true")\n    args = p.parse_args()\n    if args.spectral_aug_copies < 0:\n        raise ValueError("spectral_aug_copies must be non-negative")\n    if args.spectral_preprocess == "SP1" and args.mnf_components is None:\n        raise ValueError("SP1 requires --mnf-components selected on validation")\n    if not 0 <= args.multi_scale <= 0.9:\n        raise ValueError("multi_scale must be in 0..0.9")\n    if args.tta and args.family != "yolo":\n        raise ValueError("Built-in validated TTA is enabled only for YOLO in this runner")\n    import torch\n    if args.device != "cpu" and not torch.cuda.is_available():\n        raise RuntimeError("GPU is unavailable. Enable a Kaggle GPU and keep its CUDA PyTorch installation.")\n\n    random.seed(args.seed)\n    np.random.seed(args.seed)\n    Path(args.workdir).mkdir(parents=True, exist_ok=True)\n\n    versions = {name: importlib.metadata.version(name) for name in ("torch", "ultralytics", "numpy")}\n    run_config = {\n        "args": vars(args),\n        "resolved_input_policy": resolve_input_policy(args.family, args.input_policy),\n        "versions": versions,\n    }\n    (Path(args.workdir) / "run_config.json").write_text(json.dumps(run_config, indent=2))\n    yaml_path, test_out, inference_geometry = build_dataset(args)\n    if args.checkpoint:\n        base_checkpoint = Path(args.checkpoint)\n        if not base_checkpoint.exists():\n            raise FileNotFoundError(base_checkpoint)\n    else:\n        base_checkpoint = train(args, yaml_path)\n    base_report = evaluate(\n        args,\n        base_checkpoint,\n        yaml_path,\n        ["val"],\n        "evaluation_base.json",\n    )\n    selected_checkpoint = base_checkpoint\n    selected_report = base_report\n    selection = {"base_checkpoint": str(base_checkpoint), "localization_finetune": None, "tta": None}\n    finetuned_checkpoint = localization_finetune(args, yaml_path, base_checkpoint)\n    if finetuned_checkpoint:\n        finetuned_report = evaluate(\n            args,\n            finetuned_checkpoint,\n            yaml_path,\n            ["val"],\n            "evaluation_localized.json",\n        )\n        decision = promote_candidate(base_report, finetuned_report, args.promotion_min_gain)\n        decision["checkpoint"] = str(finetuned_checkpoint)\n        selection["localization_finetune"] = decision\n        if decision["passed"]:\n            selected_checkpoint = finetuned_checkpoint\n            selected_report = finetuned_report\n    selected_tta = False\n    if args.tta:\n        tta_report = evaluate(\n            args,\n            selected_checkpoint,\n            yaml_path,\n            ["val"],\n            "evaluation_tta.json",\n            tta=True,\n        )\n        decision = promote_candidate(selected_report, tta_report, args.tta_min_gain)\n        selection["tta"] = decision\n        if decision["passed"]:\n            selected_report = tta_report\n            selected_tta = True\n    selection["selected_checkpoint"] = str(selected_checkpoint)\n    selection["selected_checkpoint_sha256"] = hashlib.sha256(Path(selected_checkpoint).read_bytes()).hexdigest()\n    selection["selected_tta"] = selected_tta\n    (Path(args.workdir) / "selection.json").write_text(json.dumps(selection, indent=2))\n    if args.design_only:\n        print(selection)\n        print("Design-only run complete. Holdout and competition test prediction were not used.")\n        return\n    holdout_report = evaluate(\n        args,\n        selected_checkpoint,\n        yaml_path,\n        ["test"],\n        "evaluation_holdout.json",\n        tta=selected_tta,\n    )\n    report = {\n        "checkpoint": str(selected_checkpoint),\n        "sha256": selection["selected_checkpoint_sha256"],\n        "tta": selected_tta,\n        "val": selected_report["val"],\n        "test": holdout_report["test"],\n    }\n    (Path(args.workdir) / "evaluation.json").write_text(json.dumps(report, indent=2))\n    gate = check_submission_gate(args, report)\n    print(gate)\n    if not gate["passed"]:\n        print("Submission blocked: holdout mAP50-95 is below minimum.")\n        print(selected_checkpoint)\n        return\n    submission = predict(args, selected_checkpoint, test_out, inference_geometry, selected_tta)\n    print(selected_checkpoint)\n    print(submission)\n\n\nif __name__ == "__main__":\n    main()\n')
Path('/kaggle/working/evaluate_size_ap.py').write_text('import argparse\nimport contextlib\nimport copy\nimport importlib.metadata\nimport io\nimport json\nimport math\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport numpy as np\n\n\nAREA_LABELS = ("all", "small", "medium", "large")\nOFFICIAL_SOURCE = "https://github.com/cocodataset/cocoapi/blob/master/PythonAPI/pycocotools/cocoeval.py"\n\n\ndef _prepare_inputs(coco_gt, predictions):\n    gt = copy.deepcopy(coco_gt)\n    predictions = copy.deepcopy(list(predictions))\n    for key in ("images", "categories", "annotations"):\n        if key not in gt:\n            raise ValueError(f"COCO ground truth is missing {key}")\n    image_ids = [image["id"] for image in gt["images"]]\n    category_ids = [category["id"] for category in gt["categories"]]\n    if not image_ids or not category_ids:\n        raise ValueError("At least one image and category are required")\n    if len(set(image_ids)) != len(image_ids) or len(set(category_ids)) != len(category_ids):\n        raise ValueError("Image and category IDs must be unique")\n    if any(not isinstance(i, int) or isinstance(i, bool) for i in image_ids + category_ids):\n        raise ValueError("Image and category IDs must be integers")\n    for image in gt["images"]:\n        if any(not math.isfinite(float(image.get(k, 0))) or image.get(k, 0) <= 0 for k in ("width", "height")):\n            raise ValueError("Original image width and height must be finite and positive")\n    supplied_areas_replaced = 0\n    for is_prediction, rows in ((False, gt["annotations"]), (True, predictions)):\n        for index, row in enumerate(rows, 1):\n            if row["image_id"] not in image_ids or row["category_id"] not in category_ids:\n                raise ValueError("Annotation or prediction references an unknown image/category")\n            bbox = np.asarray(row.get("bbox", []), dtype=np.float64)\n            if bbox.shape != (4,) or not np.isfinite(bbox).all() or (bbox[2:] <= 0).any():\n                raise ValueError("Bboxes must be finite original-pixel XYWH with positive width/height")\n            area = float(bbox[2] * bbox[3])\n            if not math.isfinite(area):\n                raise ValueError("BBox area must be finite")\n            row["bbox"] = bbox.tolist()\n            if is_prediction:\n                score = float(row["score"])\n                if not math.isfinite(score) or not 0 <= score <= 1:\n                    raise ValueError("Prediction scores must be finite and between zero and one")\n                row["score"] = score\n            else:\n                supplied_areas_replaced += int("area" in row and row["area"] != area)\n                row["area"] = area\n                row["id"] = index\n                row["iscrowd"] = int(bool(row.get("iscrowd", 0)))\n                if row.get("ignore", 0) and not row["iscrowd"]:\n                    raise ValueError("Non-crowd ignore annotations need an explicit scoring policy")\n    gt.setdefault("info", {})\n    return gt, predictions, supplied_areas_replaced\n\n\ndef _mean_available(values):\n    values = np.asarray(values)\n    available = values[values > -1]\n    return float(available.mean()) if available.size else None\n\n\ndef _metric_row(evaluator, area_index, class_index=None):\n    precision = evaluator.eval["precision"][:, :, :, area_index, -1]\n    recall = evaluator.eval["recall"][:, :, area_index, -1]\n    if class_index is not None:\n        precision = precision[:, :, class_index]\n        recall = recall[:, class_index]\n    thresholds = evaluator.params.iouThrs\n    return {\n        "AP50_95": _mean_available(precision),\n        "AP50": _mean_available(precision[np.isclose(thresholds, 0.5)]),\n        "AP75": _mean_available(precision[np.isclose(thresholds, 0.75)]),\n        "AR50_95": _mean_available(recall),\n        "recall50": _mean_available(recall[np.isclose(thresholds, 0.5)]),\n        "recall75": _mean_available(recall[np.isclose(thresholds, 0.75)]),\n    }\n\n\ndef _evaluate(coco_gt, coco_dt, *, strict, max_dets):\n    from pycocotools.cocoeval import COCOeval\n\n    evaluator = COCOeval(coco_gt, coco_dt, "bbox")\n    evaluator.params.maxDets = sorted(set((1, min(10, max_dets), max_dets)))\n    if strict:\n        evaluator.params.areaRng = [\n            [0, float("inf")],\n            [0, float(np.nextafter(1024.0, -np.inf))],\n            [1024, float(np.nextafter(9216.0, -np.inf))],\n            [9216, float("inf")],\n        ]\n    evaluator.evaluate()\n    evaluator.accumulate()\n    categories = {category["id"]: category["name"] for category in coco_gt.dataset["categories"]}\n    return {\n        "size_metrics": {label: _metric_row(evaluator, i) for i, label in enumerate(AREA_LABELS)},\n        "class_by_size": [\n            {"category_id": int(category_id), "category_name": categories[category_id],\n             "sizes": {label: _metric_row(evaluator, i, k) for i, label in enumerate(AREA_LABELS)}}\n            for k, category_id in enumerate(evaluator.params.catIds)\n        ],\n    }\n\n\ndef evaluate_size_ap(coco_gt, predictions, *, max_dets=100):\n    from pycocotools.coco import COCO\n\n    if not isinstance(max_dets, int) or isinstance(max_dets, bool) or max_dets < 1:\n        raise ValueError("max_dets must be a positive integer")\n    gt, predictions, replaced = _prepare_inputs(coco_gt, predictions)\n    with contextlib.redirect_stdout(io.StringIO()):\n        ground_truth = COCO()\n        ground_truth.dataset = gt\n        ground_truth.createIndex()\n        if predictions:\n            detections = ground_truth.loadRes(predictions)\n        else:\n            detections = COCO()\n            detections.dataset = {"images": copy.deepcopy(gt["images"]),\n                                  "categories": copy.deepcopy(gt["categories"]), "annotations": []}\n            detections.createIndex()\n        standard = _evaluate(ground_truth, detections, strict=False, max_dets=max_dets)\n        strict = _evaluate(ground_truth, detections, strict=True, max_dets=max_dets)\n    raw_areas = [{"annotation_id": row["id"], "image_id": row["image_id"],\n                  "category_id": row["category_id"], "area": row["area"], "iscrowd": row["iscrowd"]}\n                 for row in gt["annotations"]]\n    bins = [0, 64, 256, 1024, 4096, 9216, 16384, 65536, float("inf")]\n    histogram = []\n    for lower, upper in zip(bins[:-1], bins[1:]):\n        rows = [row for row in raw_areas if lower <= row["area"] < upper]\n        histogram.append({"lower_inclusive": lower, "upper_exclusive": upper if math.isfinite(upper) else None,\n                          "count": len(rows), "non_crowd_count": sum(not row["iscrowd"] for row in rows)})\n    return {\n        "coordinate_space": "original_image_pixels_xywh",\n        "ground_truth_area_policy": "bbox_width_times_height_without_rescaling",\n        "supplied_gt_areas_replaced": replaced,\n        "pycocotools_version": importlib.metadata.version("pycocotools"),\n        "evaluator_source": OFFICIAL_SOURCE,\n        "iou_thresholds": [round(0.5 + 0.05 * i, 2) for i in range(10)],\n        "max_detections_per_image_per_class": max_dets,\n        "confidence_filter": None,\n        "boundary_policy": {\n            "roadmap_strict": "small [0,1024), medium [1024,9216), large [9216,infinity)",\n            "coco_standard": "small [0,1024], medium [1024,9216], large [9216,1e10]; inclusive overlapping boundaries",\n        },\n        "roadmap_strict": strict,\n        "coco_standard": standard,\n        "raw_box_areas": raw_areas,\n        "box_area_histogram": histogram,\n    }\n\n\ndef _iou(first, second):\n    x = max(first[0], second[0])\n    y = max(first[1], second[1])\n    right = min(first[0] + first[2], second[0] + second[2])\n    bottom = min(first[1] + first[3], second[1] + second[3])\n    intersection = max(0.0, right - x) * max(0.0, bottom - y)\n    return intersection / (first[2] * first[3] + second[2] * second[3] - intersection)\n\n\ndef detection_confusion(coco_gt, predictions, *, confidence_threshold, iou_threshold):\n    if not math.isfinite(confidence_threshold) or not 0 <= confidence_threshold <= 1:\n        raise ValueError("confidence_threshold must be between zero and one")\n    if not math.isfinite(iou_threshold) or not 0 < iou_threshold <= 1:\n        raise ValueError("iou_threshold must be greater than zero and at most one")\n    gt, predictions, _ = _prepare_inputs(coco_gt, predictions)\n    if any(row["iscrowd"] for row in gt["annotations"]):\n        raise ValueError("This independent confusion diagnostic does not support crowd annotations")\n    categories = sorted(gt["categories"], key=lambda category: category["id"])\n    indexes = {category["id"]: i for i, category in enumerate(categories)}\n    background = len(categories)\n    matrix = np.zeros((background + 1, background + 1), dtype=np.int64)\n    truths_by_image = defaultdict(list)\n    predictions_by_image = defaultdict(list)\n    for row in gt["annotations"]:\n        truths_by_image[row["image_id"]].append(row)\n    for row in predictions:\n        if row["score"] >= confidence_threshold:\n            predictions_by_image[row["image_id"]].append(row)\n    for image in gt["images"]:\n        truths = truths_by_image[image["id"]]\n        unmatched = set(range(len(truths)))\n        for prediction in sorted(predictions_by_image[image["id"]], key=lambda row: -row["score"]):\n            matches = [(i, _iou(prediction["bbox"], truths[i]["bbox"])) for i in sorted(unmatched)]\n            best = max(matches, key=lambda match: match[1]) if matches else None\n            predicted_class = indexes[prediction["category_id"]]\n            if best is not None and best[1] >= iou_threshold:\n                true_index = best[0]\n                unmatched.remove(true_index)\n                matrix[indexes[truths[true_index]["category_id"]], predicted_class] += 1\n            else:\n                matrix[background, predicted_class] += 1\n        for index in unmatched:\n            matrix[indexes[truths[index]["category_id"]], background] += 1\n    per_class = []\n    for index, category in enumerate(categories):\n        true_positive = int(matrix[index, index])\n        predicted_count = int(matrix[:, index].sum())\n        true_count = int(matrix[index, :].sum())\n        per_class.append({"category_id": category["id"], "category_name": category["name"],\n                          "true_positives": true_positive, "false_positives": predicted_count - true_positive,\n                          "false_negatives": true_count - true_positive,\n                          "precision": true_positive / predicted_count if predicted_count else None,\n                          "recall": true_positive / true_count if true_count else None})\n    return {"matrix": matrix.tolist(), "orientation": "rows=true, columns=predicted",\n            "category_ids": [category["id"] for category in categories] + [None],\n            "labels": [category["name"] for category in categories] + ["background"],\n            "confidence_threshold": confidence_threshold, "iou_threshold": iou_threshold,\n            "matching_policy": "descending score, class-agnostic highest-IoU unmatched GT; stable input-order ties",\n            "per_class": per_class}\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--ground-truth", required=True, type=Path)\n    parser.add_argument("--predictions", required=True, type=Path)\n    parser.add_argument("--output", required=True, type=Path)\n    parser.add_argument("--max-dets", type=int, default=100)\n    parser.add_argument("--confusion-confidence", type=float)\n    parser.add_argument("--confusion-iou", type=float)\n    args = parser.parse_args()\n    if (args.confusion_confidence is None) != (args.confusion_iou is None):\n        parser.error("Provide both confusion thresholds or neither")\n    gt = json.loads(args.ground_truth.read_text())\n    predictions = json.loads(args.predictions.read_text())\n    report = evaluate_size_ap(gt, predictions, max_dets=args.max_dets)\n    if args.confusion_confidence is not None:\n        report["confusion"] = detection_confusion(gt, predictions, confidence_threshold=args.confusion_confidence,\n                                                  iou_threshold=args.confusion_iou)\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    args.output.write_text(json.dumps(report, indent=2, allow_nan=False) + "\\n")\n\n\nif __name__ == "__main__":\n    main()\n')
Path('/kaggle/working/inference_common.py').write_text('from dataclasses import dataclass\n\nimport numpy as np\n\n\nN_BANDS = 16\n\n\n@dataclass(frozen=True)\nclass PredictionBatch:\n    boxes: np.ndarray\n    scores: np.ndarray\n    labels: np.ndarray\n    checkpoint_id: str\n    source: str = "model_output"\n    prediction_stage: str = "post_suppression"\n\n    def __post_init__(self):\n        boxes = np.asarray(self.boxes, dtype=np.float64)\n        scores = np.asarray(self.scores, dtype=np.float64)\n        labels = np.asarray(self.labels)\n        if boxes.ndim != 2 or boxes.shape[1:] != (4,):\n            raise ValueError("boxes must have shape (N, 4) in XYXY coordinates")\n        if scores.shape != (len(boxes),) or labels.shape != (len(boxes),):\n            raise ValueError("scores and labels must have shape (N,)")\n        if not np.isfinite(boxes).all() or not np.isfinite(scores).all():\n            raise ValueError("boxes and scores must be finite")\n        if len(boxes) and ((boxes[:, 2] <= boxes[:, 0]).any() or (boxes[:, 3] <= boxes[:, 1]).any()):\n            raise ValueError("every XYXY box must have positive width and height")\n        if ((scores < 0) | (scores > 1)).any():\n            raise ValueError("scores must be between zero and one")\n        if not np.issubdtype(labels.dtype, np.integer):\n            if not np.issubdtype(labels.dtype, np.number) or not np.isfinite(labels).all() or not np.equal(labels, np.floor(labels)).all():\n                raise ValueError("labels must be finite integer class IDs")\n        if not isinstance(self.checkpoint_id, str) or not self.checkpoint_id.strip():\n            raise ValueError("checkpoint_id must be a non-empty immutable identifier or hash")\n        if self.prediction_stage not in {"raw_pre_suppression", "post_suppression"}:\n            raise ValueError("prediction_stage must be raw_pre_suppression or post_suppression")\n        boxes = boxes.copy()\n        scores = scores.copy()\n        labels = labels.astype(np.int64, copy=True)\n        boxes.setflags(write=False)\n        scores.setflags(write=False)\n        labels.setflags(write=False)\n        object.__setattr__(self, "boxes", boxes)\n        object.__setattr__(self, "scores", scores)\n        object.__setattr__(self, "labels", labels)\n\n    def __len__(self):\n        return len(self.boxes)\n\n    def subset(self, indexes, *, source=None, prediction_stage=None, scores=None):\n        indexes = np.asarray(indexes, dtype=np.int64)\n        return PredictionBatch(\n            self.boxes[indexes],\n            self.scores[indexes] if scores is None else scores,\n            self.labels[indexes],\n            self.checkpoint_id,\n            self.source if source is None else source,\n            self.prediction_stage if prediction_stage is None else prediction_stage,\n        )\n\n\n@dataclass(frozen=True)\nclass MergeResult:\n    predictions: PredictionBatch\n    input_predictions: int\n    kept_predictions: int\n    duplicate_predictions: int\n    clipped_predictions: int = 0\n    discarded_predictions: int = 0\n\n\ndef validate_hyperspectral_cube(cube):\n    array = np.asarray(cube)\n    if array.ndim != 3 or array.shape[-1] != N_BANDS:\n        raise ValueError("hyperspectral cubes must have shape (height, width, 16)")\n    if not np.issubdtype(array.dtype, np.number) or not np.isfinite(array).all():\n        raise ValueError("hyperspectral cubes must contain finite numeric values")\n    if min(array.shape[:2]) < 1:\n        raise ValueError("hyperspectral cubes must have positive height and width")\n    return array\n\n\ndef require_same_checkpoint(batches):\n    batches = tuple(batches)\n    if not batches:\n        raise ValueError("at least one prediction batch is required")\n    checkpoint_ids = {batch.checkpoint_id for batch in batches}\n    if len(checkpoint_ids) != 1:\n        raise ValueError("predictions from different checkpoints cannot be combined")\n    return next(iter(checkpoint_ids))\n\n\ndef concatenate_same_checkpoint(batches, *, source, prediction_stage=None):\n    batches = tuple(batches)\n    checkpoint_id = require_same_checkpoint(batches)\n    stages = {batch.prediction_stage for batch in batches}\n    if prediction_stage is None:\n        if len(stages) != 1:\n            raise ValueError("prediction batches from different suppression stages cannot be combined")\n        prediction_stage = next(iter(stages))\n    return PredictionBatch(\n        np.concatenate([batch.boxes for batch in batches], axis=0),\n        np.concatenate([batch.scores for batch in batches], axis=0),\n        np.concatenate([batch.labels for batch in batches], axis=0),\n        checkpoint_id,\n        source,\n        prediction_stage,\n    )\n\n\ndef box_iou_one_to_many(box, boxes):\n    boxes = np.asarray(boxes, dtype=np.float64)\n    if len(boxes) == 0:\n        return np.empty(0, dtype=np.float64)\n    upper_left = np.maximum(box[:2], boxes[:, :2])\n    lower_right = np.minimum(box[2:], boxes[:, 2:])\n    intersection = np.prod(np.maximum(0.0, lower_right - upper_left), axis=1)\n    first_area = (box[2] - box[0]) * (box[3] - box[1])\n    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])\n    return intersection / np.maximum(first_area + areas - intersection, np.finfo(np.float64).eps)\n\n\ndef clip_prediction_batch(batch, image_width, image_height, *, source=None):\n    if not np.isfinite([image_width, image_height]).all() or image_width <= 0 or image_height <= 0:\n        raise ValueError("image dimensions must be finite and positive")\n    boxes = batch.boxes.copy()\n    bounded = np.clip(boxes, [0, 0, 0, 0], [image_width, image_height, image_width, image_height])\n    clipped = int(np.any(boxes != bounded, axis=1).sum())\n    valid = (bounded[:, 2] > bounded[:, 0]) & (bounded[:, 3] > bounded[:, 1])\n    indexes = np.flatnonzero(valid)\n    return (\n        PredictionBatch(\n            bounded[indexes],\n            batch.scores[indexes],\n            batch.labels[indexes],\n            batch.checkpoint_id,\n            batch.source if source is None else source,\n            batch.prediction_stage,\n        ),\n        clipped,\n        int((~valid).sum()),\n    )\n\n\ndef hard_nms_indices(boxes, scores, labels, iou_threshold=0.5, class_agnostic=False):\n    if not np.isfinite(iou_threshold) or not 0 <= iou_threshold <= 1:\n        raise ValueError("iou_threshold must be between zero and one")\n    boxes = np.asarray(boxes, dtype=np.float64)\n    scores = np.asarray(scores, dtype=np.float64)\n    labels = np.asarray(labels, dtype=np.int64)\n    kept = []\n    groups = [np.arange(len(boxes))] if class_agnostic else [np.flatnonzero(labels == label) for label in np.unique(labels)]\n    for group in groups:\n        order = group[np.argsort(-scores[group], kind="stable")]\n        while len(order):\n            current = int(order[0])\n            kept.append(current)\n            if len(order) == 1:\n                break\n            remaining = order[1:]\n            order = remaining[box_iou_one_to_many(boxes[current], boxes[remaining]) <= iou_threshold]\n    return np.asarray(sorted(kept, key=lambda index: (-scores[index], index)), dtype=np.int64)\n\n\ndef merge_predictions(batches, *, iou_threshold=0.5, class_agnostic=False, max_detections=None, source="same_checkpoint_merge"):\n    combined = concatenate_same_checkpoint(batches, source=source, prediction_stage="post_suppression")\n    keep = hard_nms_indices(combined.boxes, combined.scores, combined.labels, iou_threshold, class_agnostic)\n    if max_detections is not None:\n        if not isinstance(max_detections, int) or isinstance(max_detections, bool) or max_detections < 1:\n            raise ValueError("max_detections must be a positive integer")\n        keep = keep[:max_detections]\n    predictions = combined.subset(keep, source=source, prediction_stage="post_suppression")\n    return MergeResult(predictions, len(combined), len(predictions), len(combined) - len(predictions))\n')
Path('/kaggle/working/inference_multiscale.py').write_text('from dataclasses import dataclass\n\nimport numpy as np\n\nfrom inference_common import MergeResult, PredictionBatch, clip_prediction_batch, merge_predictions, validate_hyperspectral_cube\n\n\n@dataclass(frozen=True)\nclass MultiScalePlan:\n    scales: tuple[float, ...]\n    policy_id: str\n    selected_on: str = "validation_cv"\n    frozen: bool = True\n\n    def __post_init__(self):\n        scales = tuple(float(scale) for scale in self.scales)\n        if len(scales) < 2 or not np.isfinite(scales).all() or any(scale <= 0 for scale in scales):\n            raise ValueError("a multi-scale plan requires at least two finite positive scales")\n        if len(set(scales)) != len(scales):\n            raise ValueError("multi-scale plan scales must be unique")\n        if not isinstance(self.policy_id, str) or not self.policy_id.strip():\n            raise ValueError("policy_id must be a non-empty predeclared policy identifier")\n        if self.selected_on not in {"validation", "cv", "validation_cv"}:\n            raise ValueError("multi-scale policies must be selected on validation/CV, never holdout or test")\n        if self.frozen is not True:\n            raise ValueError("the multi-scale policy must be frozen before evaluation")\n        object.__setattr__(self, "scales", scales)\n\n\ndef scaled_shape(original_shape, scale):\n    if len(original_shape) != 2:\n        raise ValueError("original_shape must be (height, width)")\n    height, width = original_shape\n    if not isinstance(height, int) or not isinstance(width, int) or min(height, width) < 1:\n        raise ValueError("original dimensions must be positive integers")\n    if not np.isfinite(scale) or scale <= 0:\n        raise ValueError("scale must be finite and positive")\n    return max(1, int(round(height * scale))), max(1, int(round(width * scale)))\n\n\ndef scale_cube(cube, scale):\n    array = validate_hyperspectral_cube(cube)\n    target_height, target_width = scaled_shape(array.shape[:2], scale)\n    if (target_height, target_width) == array.shape[:2]:\n        return array.copy()\n    source_y = np.clip((np.arange(target_height) + 0.5) * array.shape[0] / target_height - 0.5, 0, array.shape[0] - 1)\n    source_x = np.clip((np.arange(target_width) + 0.5) * array.shape[1] / target_width - 0.5, 0, array.shape[1] - 1)\n    y0 = np.floor(source_y).astype(np.int64)\n    x0 = np.floor(source_x).astype(np.int64)\n    y1 = np.minimum(y0 + 1, array.shape[0] - 1)\n    x1 = np.minimum(x0 + 1, array.shape[1] - 1)\n    wy = (source_y - y0)[:, None, None]\n    wx = (source_x - x0)[None, :, None]\n    top = array[y0[:, None], x0[None, :]] * (1 - wx) + array[y0[:, None], x1[None, :]] * wx\n    bottom = array[y1[:, None], x0[None, :]] * (1 - wx) + array[y1[:, None], x1[None, :]] * wx\n    dtype = array.dtype if np.issubdtype(array.dtype, np.floating) else np.float32\n    return (top * (1 - wy) + bottom * wy).astype(dtype, copy=False)\n\n\ndef restore_scaled_predictions(predictions, original_shape, scaled_image_shape, *, source=None):\n    original_height, original_width = original_shape\n    scaled_height, scaled_width = scaled_image_shape\n    if min(original_height, original_width, scaled_height, scaled_width) < 1:\n        raise ValueError("image dimensions must be positive")\n    scale_x = scaled_width / original_width\n    scale_y = scaled_height / original_height\n    boxes = predictions.boxes / np.asarray([scale_x, scale_y, scale_x, scale_y])\n    restored = PredictionBatch(\n        boxes,\n        predictions.scores,\n        predictions.labels,\n        predictions.checkpoint_id,\n        source or f"{predictions.source}:scale_restored",\n        predictions.prediction_stage,\n    )\n    return clip_prediction_batch(restored, original_width, original_height)\n\n\ndef restore_multiscale_predictions(\n    predictions_by_scale,\n    original_shape,\n    plan,\n    *,\n    iou_threshold=0.5,\n    class_agnostic=False,\n    max_detections=None,\n):\n    if not isinstance(plan, MultiScalePlan):\n        raise TypeError("plan must be a frozen MultiScalePlan")\n    if set(predictions_by_scale) != set(plan.scales):\n        raise ValueError("prediction scales must exactly match the predeclared multi-scale plan")\n    restored = []\n    clipped = 0\n    discarded = 0\n    for scale in plan.scales:\n        shape = scaled_shape(original_shape, scale)\n        batch, batch_clipped, batch_discarded = restore_scaled_predictions(\n            predictions_by_scale[scale], original_shape, shape, source=f"multiscale:{plan.policy_id}:{scale:g}"\n        )\n        restored.append(batch)\n        clipped += batch_clipped\n        discarded += batch_discarded\n    merged = merge_predictions(\n        restored,\n        iou_threshold=iou_threshold,\n        class_agnostic=class_agnostic,\n        max_detections=max_detections,\n        source=f"T2:{plan.policy_id}",\n    )\n    return MergeResult(\n        merged.predictions,\n        merged.input_predictions + discarded,\n        merged.kept_predictions,\n        merged.duplicate_predictions,\n        clipped,\n        discarded,\n    )\n')
Path('/kaggle/working/inference_sliced.py').write_text('from dataclasses import dataclass\n\nimport numpy as np\n\nfrom inference_common import MergeResult, PredictionBatch, clip_prediction_batch, merge_predictions, validate_hyperspectral_cube\n\n\n@dataclass(frozen=True)\nclass SliceWindow:\n    x1: int\n    y1: int\n    x2: int\n    y2: int\n    index: int\n\n    @property\n    def width(self):\n        return self.x2 - self.x1\n\n    @property\n    def height(self):\n        return self.y2 - self.y1\n\n\n@dataclass(frozen=True)\nclass SlicedInferenceReport:\n    result: MergeResult\n    windows: tuple[SliceWindow, ...]\n    checkpoint_id: str\n    raw_predictions: int\n\n\ndef _axis_starts(length, window, overlap):\n    if not isinstance(length, int) or not isinstance(window, int) or min(length, window) < 1:\n        raise ValueError("image and slice dimensions must be positive integers")\n    if not np.isfinite(overlap) or not 0 <= overlap < 1:\n        raise ValueError("overlap fractions must be in [0, 1)")\n    window = min(length, window)\n    stride = max(1, int(round(window * (1 - overlap))))\n    starts = list(range(0, max(length - window, 0) + 1, stride))\n    final = length - window\n    if not starts or starts[-1] != final:\n        starts.append(final)\n    return tuple(dict.fromkeys(starts)), window\n\n\ndef generate_slice_windows(image_shape, slice_shape, overlap=(0.2, 0.2)):\n    if len(image_shape) != 2 or len(slice_shape) != 2 or len(overlap) != 2:\n        raise ValueError("image_shape, slice_shape and overlap must each have two values")\n    image_height, image_width = image_shape\n    slice_height, slice_width = slice_shape\n    y_starts, actual_height = _axis_starts(image_height, slice_height, float(overlap[0]))\n    x_starts, actual_width = _axis_starts(image_width, slice_width, float(overlap[1]))\n    return tuple(\n        SliceWindow(x, y, x + actual_width, y + actual_height, index)\n        for index, (y, x) in enumerate((y, x) for y in y_starts for x in x_starts)\n    )\n\n\ndef extract_slice(cube, window):\n    array = validate_hyperspectral_cube(cube)\n    if not isinstance(window, SliceWindow):\n        raise TypeError("window must be a SliceWindow")\n    if window.x1 < 0 or window.y1 < 0 or window.x2 > array.shape[1] or window.y2 > array.shape[0]:\n        raise ValueError("slice window lies outside the image")\n    return array[window.y1:window.y2, window.x1:window.x2, :].copy()\n\n\ndef restore_slice_predictions(predictions, window, image_shape):\n    image_height, image_width = image_shape\n    local, local_clipped, local_discarded = clip_prediction_batch(predictions, window.width, window.height)\n    boxes = local.boxes + np.asarray([window.x1, window.y1, window.x1, window.y1])\n    restored = PredictionBatch(\n        boxes,\n        local.scores,\n        local.labels,\n        local.checkpoint_id,\n        f"{local.source}:slice_{window.index}_restored",\n        local.prediction_stage,\n    )\n    bounded, global_clipped, global_discarded = clip_prediction_batch(restored, image_width, image_height)\n    return bounded, local_clipped + global_clipped, local_discarded + global_discarded\n\n\ndef merge_sliced_predictions(\n    predictions_by_window,\n    windows,\n    image_shape,\n    *,\n    checkpoint_id,\n    iou_threshold=0.5,\n    class_agnostic=False,\n    max_detections=None,\n):\n    windows = tuple(windows)\n    if set(predictions_by_window) != {window.index for window in windows}:\n        raise ValueError("predictions must contain exactly one batch for every declared slice window")\n    restored = []\n    raw = 0\n    clipped = 0\n    discarded = 0\n    for window in windows:\n        batch = predictions_by_window[window.index]\n        if batch.checkpoint_id != checkpoint_id:\n            raise ValueError("sliced inference cannot combine predictions from different checkpoints")\n        raw += len(batch)\n        transformed, batch_clipped, batch_discarded = restore_slice_predictions(batch, window, image_shape)\n        restored.append(transformed)\n        clipped += batch_clipped\n        discarded += batch_discarded\n    merged = merge_predictions(\n        restored,\n        iou_threshold=iou_threshold,\n        class_agnostic=class_agnostic,\n        max_detections=max_detections,\n        source="T4/P3:sliced_inference",\n    )\n    result = MergeResult(\n        merged.predictions,\n        raw,\n        merged.kept_predictions,\n        merged.duplicate_predictions,\n        clipped,\n        discarded,\n    )\n    return SlicedInferenceReport(result, windows, checkpoint_id, raw)\n\n\ndef run_sliced_inference(\n    cube,\n    predictor,\n    checkpoint_id,\n    *,\n    slice_shape,\n    overlap=(0.2, 0.2),\n    iou_threshold=0.5,\n    class_agnostic=False,\n    max_detections=None,\n):\n    array = validate_hyperspectral_cube(cube)\n    if not callable(predictor):\n        raise TypeError("predictor must be callable as predictor(slice_cube, slice_window)")\n    windows = generate_slice_windows(array.shape[:2], slice_shape, overlap)\n    predictions = {}\n    for window in windows:\n        batch = predictor(extract_slice(array, window), window)\n        if not isinstance(batch, PredictionBatch):\n            raise TypeError("predictor must return PredictionBatch")\n        predictions[window.index] = batch\n    return merge_sliced_predictions(\n        predictions,\n        windows,\n        array.shape[:2],\n        checkpoint_id=checkpoint_id,\n        iou_threshold=iou_threshold,\n        class_agnostic=class_agnostic,\n        max_detections=max_detections,\n    )\n')
Path('/kaggle/working/inference_tta.py').write_text('import numpy as np\n\nfrom inference_common import MergeResult, PredictionBatch, concatenate_same_checkpoint, merge_predictions, validate_hyperspectral_cube\nfrom inference_multiscale import MultiScalePlan, restore_multiscale_predictions, scaled_shape\n\n\ndef horizontal_flip_cube(cube):\n    array = validate_hyperspectral_cube(cube)\n    return np.flip(array, axis=1).copy()\n\n\ndef restore_horizontal_boxes(boxes, image_width):\n    boxes = np.asarray(boxes, dtype=np.float64)\n    if boxes.ndim != 2 or boxes.shape[1:] != (4,) or not np.isfinite(boxes).all():\n        raise ValueError("boxes must be a finite (N, 4) XYXY array")\n    if not np.isfinite(image_width) or image_width <= 0:\n        raise ValueError("image_width must be finite and positive")\n    restored = boxes.copy()\n    restored[:, 0] = image_width - boxes[:, 2]\n    restored[:, 2] = image_width - boxes[:, 0]\n    if len(restored) and ((restored[:, 2] <= restored[:, 0]).any() or (restored[:, 3] <= restored[:, 1]).any()):\n        raise ValueError("restored boxes must have positive width and height")\n    return restored\n\n\ndef restore_horizontal_predictions(predictions, image_width):\n    return PredictionBatch(\n        restore_horizontal_boxes(predictions.boxes, image_width),\n        predictions.scores,\n        predictions.labels,\n        predictions.checkpoint_id,\n        f"{predictions.source}:horizontal_restored",\n        predictions.prediction_stage,\n    )\n\n\ndef horizontal_flip_tta(\n    normal_predictions,\n    flipped_predictions,\n    image_width,\n    *,\n    iou_threshold=0.5,\n    class_agnostic=False,\n    max_detections=None,\n):\n    restored = restore_horizontal_predictions(flipped_predictions, image_width)\n    return merge_predictions(\n        (normal_predictions, restored),\n        iou_threshold=iou_threshold,\n        class_agnostic=class_agnostic,\n        max_detections=max_detections,\n        source="T1:normal_horizontal_flip",\n    )\n\n\ndef flip_multiscale_tta(\n    normal_predictions_by_scale,\n    flipped_predictions_by_scale,\n    original_shape,\n    plan,\n    *,\n    iou_threshold=0.5,\n    class_agnostic=False,\n    max_detections=None,\n):\n    if not isinstance(plan, MultiScalePlan):\n        raise TypeError("plan must be a frozen MultiScalePlan")\n    if set(normal_predictions_by_scale) != set(plan.scales) or set(flipped_predictions_by_scale) != set(plan.scales):\n        raise ValueError("normal and flipped prediction scales must exactly match the predeclared plan")\n    combined = {}\n    for scale in plan.scales:\n        scaled_width = scaled_shape(original_shape, scale)[1]\n        restored_flip = restore_horizontal_predictions(flipped_predictions_by_scale[scale], scaled_width)\n        combined[scale] = concatenate_same_checkpoint(\n            (normal_predictions_by_scale[scale], restored_flip),\n            source=f"T3:{plan.policy_id}:{scale:g}",\n            prediction_stage="post_suppression",\n        )\n    result = restore_multiscale_predictions(\n        combined,\n        original_shape,\n        plan,\n        iou_threshold=iou_threshold,\n        class_agnostic=class_agnostic,\n        max_detections=max_detections,\n    )\n    predictions = PredictionBatch(\n        result.predictions.boxes,\n        result.predictions.scores,\n        result.predictions.labels,\n        result.predictions.checkpoint_id,\n        f"T3:{plan.policy_id}",\n        "post_suppression",\n    )\n    return MergeResult(\n        predictions,\n        result.input_predictions,\n        result.kept_predictions,\n        result.duplicate_predictions,\n        result.clipped_predictions,\n        result.discarded_predictions,\n    )\n')
Path('/kaggle/working/run_inference_screen.py').write_text('from collections import defaultdict\nfrom pathlib import Path\nimport hashlib\nimport json\nimport time\n\nimport numpy as np\n\nimport hsi_runner as runner\nfrom evaluate_size_ap import detection_confusion, evaluate_size_ap\nfrom inference_common import PredictionBatch, merge_predictions\nfrom inference_multiscale import MultiScalePlan\nfrom inference_sliced import generate_slice_windows, merge_sliced_predictions\nfrom inference_tta import horizontal_flip_tta\n\n\nOUT = Path("/kaggle/working/inference_screen")\nOUT.mkdir(parents=True, exist_ok=True)\nSCALES = (512, 640, 768)\nMERGE_IOU = 0.65\nSLICE_SHAPE = (256, 256)\nSLICE_OVERLAP = (0.25, 0.25)\n\n\ndef one(root, pattern):\n    matches = list(Path(root).rglob(pattern))\n    if len(matches) != 1:\n        raise RuntimeError(f"Expected one {pattern}, found {len(matches)}")\n    return matches[0]\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef empty(checkpoint, source):\n    return PredictionBatch(np.empty((0, 4)), np.empty(0), np.empty(0, dtype=int), checkpoint, source)\n\n\ndef batch_from_result(result, checkpoint, geometry, source):\n    if result.boxes is None or len(result.boxes) == 0:\n        return empty(checkpoint, source)\n    boxes = runner.restore_boxes(result.boxes.xyxy.cpu().numpy(), geometry)\n    scores = result.boxes.conf.cpu().numpy()\n    labels = result.boxes.cls.cpu().numpy().astype(int)\n    valid = (boxes[:, 2] > boxes[:, 0]) & (boxes[:, 3] > boxes[:, 1])\n    return PredictionBatch(boxes[valid], scores[valid], labels[valid], checkpoint, source)\n\n\ndef predict_directory(model, directory, geometries, checkpoint, imgsz, source):\n    output = {}\n    for result in model.predict(source=str(directory), imgsz=imgsz, conf=0.001, iou=0.7, max_det=300,\n                                device="0", stream=True, verbose=False):\n        stem = Path(result.path).stem\n        output[stem] = batch_from_result(result, checkpoint, geometries[stem], source)\n    if set(output) != set(geometries):\n        raise RuntimeError(f"Incomplete inference for {source}")\n    return output\n\n\ndef restore_flip(batch, width):\n    if not len(batch):\n        return batch\n    boxes = batch.boxes.copy()\n    left = width - boxes[:, 2]\n    right = width - boxes[:, 0]\n    boxes[:, 0], boxes[:, 2] = left, right\n    return PredictionBatch(boxes, batch.scores, batch.labels, batch.checkpoint_id, batch.source + ":restored")\n\n\ndef coco_predictions(batches):\n    rows = []\n    for stem, batch in batches.items():\n        for box, score, label in zip(batch.boxes, batch.scores, batch.labels):\n            x1, y1, x2, y2 = map(float, box)\n            rows.append({"image_id": int(stem), "category_id": int(label), "bbox": [x1, y1, x2 - x1, y2 - y1], "score": float(score)})\n    return rows\n\n\ndef evaluate_policy(gt, batches, name):\n    predictions = coco_predictions(batches)\n    report = evaluate_size_ap(gt, predictions, max_dets=300)\n    report["confusion_at_0_25_iou_0_5"] = detection_confusion(gt, predictions, confidence_threshold=0.25, iou_threshold=0.5)\n    report["policy"] = name\n    report["prediction_count"] = len(predictions)\n    (OUT / f"{name}_predictions.json").write_text(json.dumps(predictions, allow_nan=False))\n    (OUT / f"{name}_metrics.json").write_text(json.dumps(report, indent=2, allow_nan=False))\n    return report\n\n\ndef main():\n    started = time.monotonic()\n    data = one("/kaggle/input", "hyperspectral-2026")\n    plan = one("/kaggle/input", "hsi_plan_c_ratio")\n    weights = one(plan, "best.pt")\n    expected_hash = "b4a54b0f5ad05a25fae96daf070e53286852f15c7091dd3fa13af2113afb9012"\n    if sha(weights) != expected_hash:\n        raise RuntimeError("Plan-C checkpoint hash mismatch")\n    audit = json.loads((plan / "data_audit.json").read_text())\n    split = json.loads((plan / "split_manifest.json").read_text())\n    lo = np.asarray(audit["band_low_0_5pct"], dtype=np.float32)\n    hi = np.asarray(audit["band_high_99_5pct"], dtype=np.float32)\n    names = runner.read_classes(data / "class.txt")\n    class_to_id = {name: index for index, name in enumerate(names)}\n    cubes = runner.unique_stems(runner.collect_cubes(data / "data_train/data_train/VIS"))\n    xmls = runner.annotation_map(data / "data_train/data_train/Annotations/VIS")\n    normal_dir = Path("/kaggle/temp/inference_screen/normal")\n    flip_dir = Path("/kaggle/temp/inference_screen/flip")\n    slice_dir = Path("/kaggle/temp/inference_screen/slices")\n    for directory in (normal_dir, flip_dir, slice_dir):\n        directory.mkdir(parents=True, exist_ok=False)\n    geometries = {}\n    flip_geometries = {}\n    images = []\n    annotations = []\n    original_shapes = {}\n    cube_cache = {}\n    for stem in split["val"]:\n        cube = runner.read_cube(cubes[stem])\n        width, height, rows = runner.parse_voc(xmls[stem], class_to_id)\n        original_shapes[stem] = (height, width)\n        cube_cache[stem] = cube\n        images.append({"id": int(stem), "width": width, "height": height})\n        for label, cx, cy, bw, bh in rows:\n            bbox = [(cx - bw / 2) * width, (cy - bh / 2) * height, bw * width, bh * height]\n            annotations.append({"id": len(annotations) + 1, "image_id": int(stem), "category_id": int(label),\n                                "bbox": bbox, "area": bbox[2] * bbox[3], "iscrowd": 0})\n        geometries[stem] = runner.write_tiff(cube, normal_dir / f"{stem}.tiff", lo, hi, "square_pad")\n        flip_geometries[stem] = runner.write_tiff(np.flip(cube, axis=1).copy(), flip_dir / f"{stem}.tiff", lo, hi, "square_pad")\n    gt = {"images": images, "annotations": annotations, "categories": [{"id": i, "name": name} for i, name in enumerate(names)]}\n    (OUT / "ground_truth_original.json").write_text(json.dumps(gt, indent=2))\n    from ultralytics import RTDETR\n    model = RTDETR(str(weights))\n    normal_by_scale = {}\n    flip_raw_by_scale = {}\n    for scale in SCALES:\n        normal_by_scale[scale] = predict_directory(model, normal_dir, geometries, expected_hash, scale, f"normal_{scale}")\n        flip_raw_by_scale[scale] = predict_directory(model, flip_dir, flip_geometries, expected_hash, scale, f"flip_{scale}")\n    baseline = normal_by_scale[640]\n    t1 = {stem: horizontal_flip_tta(baseline[stem], flip_raw_by_scale[640][stem], original_shapes[stem][1],\n                                    iou_threshold=MERGE_IOU, max_detections=300).predictions for stem in baseline}\n    plan_scales = MultiScalePlan(tuple(float(scale) for scale in SCALES), "512_640_768")\n    t2 = {stem: merge_predictions([normal_by_scale[scale][stem] for scale in SCALES], iou_threshold=MERGE_IOU,\n                                  max_detections=300, source=f"T2:{plan_scales.policy_id}").predictions for stem in baseline}\n    t3 = {stem: merge_predictions(\n        [batch for scale in SCALES for batch in (normal_by_scale[scale][stem], restore_flip(flip_raw_by_scale[scale][stem], original_shapes[stem][1]))],\n        iou_threshold=MERGE_IOU, max_detections=300, source=f"T3:{plan_scales.policy_id}"\n    ).predictions for stem in baseline}\n    slice_geometries = {}\n    slice_metadata = {}\n    windows_by_stem = {}\n    for stem, cube in cube_cache.items():\n        windows = generate_slice_windows(cube.shape[:2], SLICE_SHAPE, SLICE_OVERLAP)\n        windows_by_stem[stem] = windows\n        for window in windows:\n            key = f"{stem}__s{window.index}"\n            crop = cube[window.y1:window.y2, window.x1:window.x2]\n            slice_geometries[key] = runner.write_tiff(crop, slice_dir / f"{key}.tiff", lo, hi, "square_pad")\n            slice_metadata[key] = (stem, window)\n    raw_slices = predict_directory(model, slice_dir, slice_geometries, expected_hash, 640, "slice_640")\n    grouped = defaultdict(dict)\n    for key, batch in raw_slices.items():\n        stem, window = slice_metadata[key]\n        grouped[stem][window.index] = batch\n    p3 = {}\n    t4 = {}\n    slice_reports = {}\n    for stem in baseline:\n        report = merge_sliced_predictions(grouped[stem], windows_by_stem[stem], original_shapes[stem],\n                                          checkpoint_id=expected_hash, iou_threshold=MERGE_IOU, max_detections=300)\n        p3[stem] = report.result.predictions\n        t4[stem] = merge_predictions((baseline[stem], p3[stem]), iou_threshold=MERGE_IOU,\n                                     max_detections=300, source="T4:full_plus_slices").predictions\n        slice_reports[stem] = {\n            "windows": len(report.windows),\n            "raw_predictions": report.raw_predictions,\n            "kept_predictions": report.result.kept_predictions,\n            "duplicate_predictions": report.result.duplicate_predictions,\n            "clipped_predictions": report.result.clipped_predictions,\n            "discarded_predictions": report.result.discarded_predictions,\n        }\n    reports = {}\n    for name, batches in {"B0": baseline, "T1": t1, "T2": t2, "T3": t3, "P3": p3, "T4": t4}.items():\n        reports[name] = evaluate_policy(gt, batches, name)\n    reference = 0.7026892903603926\n    reproduced = reports["B0"]["roadmap_strict"]["size_metrics"]["all"]["AP50_95"]\n    if abs(reproduced - reference) > 0.005:\n        raise RuntimeError(f"Baseline reproduction drift: {reproduced} vs {reference}")\n    summary = {\n        "checkpoint_hash": expected_hash,\n        "split_hash": sha(plan / "split_manifest.json"),\n        "code_hash": sha(__file__),\n        "gpu": "Tesla T4",\n        "policies_selected_on": "legacy_validation_design_screen",\n        "submission_created": False,\n        "scales": list(SCALES),\n        "merge_iou": MERGE_IOU,\n        "slice_shape": list(SLICE_SHAPE),\n        "slice_overlap": list(SLICE_OVERLAP),\n        "runtime_seconds": time.monotonic() - started,\n        "slice_audit": slice_reports,\n        "results": {\n            name: {\n                "map50_95": report["roadmap_strict"]["size_metrics"]["all"]["AP50_95"],\n                "map75": report["roadmap_strict"]["size_metrics"]["all"]["AP75"],\n                "AP_small": report["roadmap_strict"]["size_metrics"]["small"]["AP50_95"],\n                "AP_medium": report["roadmap_strict"]["size_metrics"]["medium"]["AP50_95"],\n                "AP_large": report["roadmap_strict"]["size_metrics"]["large"]["AP50_95"],\n                "predictions": report["prediction_count"],\n            }\n            for name, report in reports.items()\n        },\n    }\n    (OUT / "summary.json").write_text(json.dumps(summary, indent=2, allow_nan=False))\n    print(json.dumps(summary, indent=2, allow_nan=False))\n\n\nif __name__ == "__main__":\n    main()\n')


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '/kaggle/working/run_inference_screen.py'], check=True)
